# 03 · Create / update the Databricks job
Run once locally (`uv sync --extra dbx`); re-run after editing 02 or the schedule.

In [ ]:
PROFILE = "corp"                              # ~/.databrickscfg
NOTEBOOK = "02_dbx_count_watch.ipynb"
WORKSPACE_DIR = None                          # None = /Users/<me>/count_watch
JOB_NAME = "count-watch"
CRON = "0 0 9,13,17 * * ?"                    # quartz: 09:00, 13:00, 17:00 (each match is emailed once)
TIMEZONE = "America/New_York"
EMAIL = ""                                    # job route: who gets "ready" and failure emails
WIDGETS = {
    "WATCH_ROOT": "/Volumes/<catalog>/<schema>/<volume>/count_watch",
    "STATUS_TABLE": "<catalog>.<schema>.count_watch_status",
    "LOOKBACK_DAYS": "7",
    "NOTIFY_VIA": "job",                      # or "smtp" (02 sends the config as an attachment)
    "SMTP_SCOPE": "count-watch",
    "RUNS_ROOT": "s3://<bucket>/<prefix>/runs",
}
RUN_NOW = False                               # True: trigger one run and wait for it

In [ ]:
from datetime import timedelta
from pathlib import Path

from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs
from databricks.sdk.service.workspace import ImportFormat

w = WorkspaceClient(profile=PROFILE)
me = w.current_user.me().user_name
folder = WORKSPACE_DIR or f"/Users/{me}/count_watch"
notebook_path = f"{folder}/{Path(NOTEBOOK).stem}"
w.workspace.mkdirs(folder)
w.workspace.upload(notebook_path, Path(NOTEBOOK).read_bytes(), format=ImportFormat.JUPYTER, overwrite=True)
print("uploaded", notebook_path)

In [ ]:
# watch -> (ready > 0 ?) -> notify; the notify task's success email is the "start validating" mail
def notebook(mode):
    return jobs.NotebookTask(notebook_path=notebook_path, base_parameters={**WIDGETS, "MODE": mode},
                             source=jobs.Source.WORKSPACE)


tasks = [jobs.Task(task_key="watch", notebook_task=notebook("watch"))]
if WIDGETS["NOTIFY_VIA"] == "job":
    tasks += [
        jobs.Task(task_key="ready", depends_on=[jobs.TaskDependency(task_key="watch")],
                  condition_task=jobs.ConditionTask(op=jobs.ConditionTaskOp.GREATER_THAN,
                                                    left="{{tasks.watch.values.ready}}", right="0")),
        jobs.Task(task_key="notify", depends_on=[jobs.TaskDependency(task_key="ready", outcome="true")],
                  notebook_task=notebook("notify"),
                  email_notifications=jobs.TaskEmailNotifications(on_success=[EMAIL]) if EMAIL else None),
    ]
settings = jobs.JobSettings(
    name=JOB_NAME,
    tasks=tasks,
    schedule=jobs.CronSchedule(quartz_cron_expression=CRON, timezone_id=TIMEZONE, pause_status=jobs.PauseStatus.UNPAUSED),
    email_notifications=jobs.JobEmailNotifications(on_failure=[EMAIL]) if EMAIL else None,
    max_concurrent_runs=1,
)
existing = [job for job in w.jobs.list(name=JOB_NAME)]
if existing:
    job_id = existing[0].job_id
    w.jobs.reset(job_id=job_id, new_settings=settings)
else:
    job_id = w.jobs.create(name=settings.name, tasks=settings.tasks, schedule=settings.schedule,
                           email_notifications=settings.email_notifications, max_concurrent_runs=1).job_id
print("job", job_id, f"{w.config.host}/jobs/{job_id}")

In [ ]:
if RUN_NOW:
    run = w.jobs.run_now(job_id=job_id).result(timeout=timedelta(minutes=30))
    for task in sorted(run.tasks, key=lambda t: t.task_key):
        print(task.task_key, task.state.result_state, task.state.state_message or "")
    watch = next(t for t in run.tasks if t.task_key == "watch")
    print(w.jobs.get_run_output(watch.run_id).notebook_output.result)